## 6 - Modelo ICB

Estima el modelo bayesiano (ICB) en primeras diferencias sobre el `PanelModelo`, con la misma
especificación del modelo frecuentista: lee las decisiones y el export del TWFE, comprime el panel
en estadísticos suficientes (Gramian distribuido) y muestrea con NUTS. Produce los efectos por
combinación con HDI 95, el bundle oficial, la convergencia formal contra el TWFE y las pruebas
de robustez bayesianas (sensibilidad a priors, PPC y ancho de HDI).

In [ ]:
pip install "jax==0.4.35" "jaxlib==0.4.35" numpyro

# Parámetros

Único bloque a editar por embotellador-país.

In [ ]:
BU = 'MEX'


In [ ]:
# NUTS
N_WARMUP      = 1500
N_SAMPLES     = 1500
N_CHAINS      = 2
TARGET_ACCEPT = 0.9
SEED          = 0
UMBRAL_RHAT   = 1.05

# Salvaguarda de identificación (espejo del modelo frecuentista)
UMBRAL_SE = 10.0

# Setup y Lectura de Datos

In [ ]:
import numpy as np
import pandas as pd
import json
import time
import matplotlib.pyplot as plt
from pyspark.sql import functions as sf, Window
from pyspark.storagelevel import StorageLevel
from pyspark.mllib.linalg import Vectors
from pyspark.mllib.linalg.distributed import RowMatrix
import jax
jax.config.update('jax_enable_x64', True)   # float32 rompe la forma cuadrática (y'y ~ 1e9)
import jax.numpy as jnp
import numpyro
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS, init_to_value, init_to_median
from numpyro.diagnostics import summary as np_summary

spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.shuffle.partitions", "200")

PATH_MODELO     = f"abfss://{containerName}@{storageAccountName}.dfs.core.windows.net/CTG/{BU}/PanelModelo/parquet/"
PATH_DECISIONES = f"abfss://{containerName}@{storageAccountName}.dfs.core.windows.net/CTG/{BU}/Modelo/capacidades"
PATH_TWFE       = f"abfss://{containerName}@{storageAccountName}.dfs.core.windows.net/CTG/{BU}/Modelo/twfe_fd"
EPS = 1

In [ ]:
# Decisiones del modelo y export del frecuentista: la especificación es una sola
rows = spark.read.text(PATH_DECISIONES).collect()
DECISIONES = json.loads('\n'.join(r[0] for r in rows))
FORMA       = DECISIONES['formas']
REZAGOS     = DECISIONES['rezagos']
LISTA_C9    = DECISIONES['combinaciones_estructura']
CAPACIDADES = list(FORMA.keys())

rows_t = spark.read.text(PATH_TWFE).collect()
TWFE = json.loads('\n'.join(r[0] for r in rows_t))
REF                  = {k: float(v) for k, v in TWFE['referencias'].items()}
REZAGOS_FUERA_BUNDLE = TWFE.get('rezagos_fuera_bundle', [])
TWFE_TERMS = TWFE['terms']
TWFE_COEF  = dict(zip(TWFE['terms'], TWFE['coef']))
TWFE_V     = np.asarray(TWFE['V'])

print(f"BU: {DECISIONES.get('bu')} | universo: {DECISIONES.get('universo')} | escala: {DECISIONES.get('convencion_proporciones')}")
print(f"Combinaciones ({len(LISTA_C9)}) | rezagos t-1: {REZAGOS} | fuera de agregados: {REZAGOS_FUERA_BUNDLE}")
print(f"Export frecuentista: {len(TWFE_TERMS)} términos ({TWFE.get('modelo')}, {TWFE.get('fecha_generacion')})")

# Construcción del diseño

In [ ]:
# Columnas derivadas según la forma de cada capacidad, términos por combinación (cuadráticas desdobladas en x y x²) y rezagos t-1 por join de calendario
CUADRATICAS = [c for c in CAPACIDADES if FORMA[c] == 'cuadratica']

def termino_forma(cap):
    return f'log_{cap}' if FORMA[cap] == 'log' else cap

caps_usadas = sorted({c for combo in LISTA_C9 for c in combo})
panel = (spark.read.parquet(PATH_MODELO)
         .select('id_cliente', 'period_id', 'ingreso_neto_core_real', *caps_usadas)
         .withColumn('log_y', sf.log(sf.greatest(sf.col('ingreso_neto_core_real'), sf.lit(0.01))))
         .filter(sf.col('log_y').isNotNull()))
for cap in caps_usadas:
    panel = panel.withColumn(cap, sf.coalesce(sf.col(cap).cast('double'), sf.lit(0.0)))
    if FORMA[cap] == 'log':
        panel = panel.withColumn(f'log_{cap}', sf.log(sf.col(cap) + sf.lit(EPS)))
    if FORMA[cap] == 'cuadratica':
        panel = panel.withColumn(f'{cap}_sq', sf.col(cap) * sf.col(cap))

def construir_terminos(combo):
    quads = [c for c in combo if c in CUADRATICAS]
    if not quads:
        return [':'.join(termino_forma(c) for c in combo)]
    q = quads[0]
    otras = [c for c in combo if c != q]
    base = ':'.join(termino_forma(c) for c in otras)
    return [f'{base}:{q}' if base else q, f'{base}:{q}_sq' if base else f'{q}_sq']

TERMINOS = []
for combo in LISTA_C9:
    for t in construir_terminos(combo):
        if t not in TERMINOS:
            TERMINOS.append(t)
for t in TERMINOS:
    if ':' in t:
        expr = sf.lit(1.0)
        for q in t.split(':'):
            expr = expr * sf.col(q)
        panel = panel.withColumn(t, expr)

# Rezagos t-1: solo capacidades con término contemporáneo (regla del rezago huérfano); join de calendario sobre period_id - 1 para no saltar meses faltantes
LAGS = []
for cap in [c for c in REZAGOS if any(c in combo for combo in LISTA_C9)]:
    col = termino_forma(cap) if FORMA[cap] != 'cuadratica' else cap
    prev = panel.select(sf.col('id_cliente'),
                        (sf.col('period_id') + 1).alias('period_id'),
                        sf.col(col).alias(f'{cap}_l1'))
    panel = panel.join(prev, ['id_cliente', 'period_id'], 'left').fillna(0.0, subset=[f'{cap}_l1'])
    LAGS.append(f'{cap}_l1')

# La especificación debe ser idéntica a la del frecuentista (mismos términos finales)
def _dname(t):
    return 'd_' + t.replace(':', '__')
NO_IDENT_TWFE = set(TWFE.get('no_identificados', []))
TERMS_ALL = [t for t in TERMINOS + LAGS if _dname(t) not in NO_IDENT_TWFE]
d_local = [_dname(t) for t in TERMS_ALL]
if d_local == list(TWFE_TERMS):
    print(f"Especificación: {len(TERMS_ALL)} términos — IDÉNTICA a la del frecuentista")
else:
    print("ATENCIÓN: la especificación difiere de la del frecuentista")
    print(f"  solo aquí:  {[t for t in d_local if t not in TWFE_TERMS]}")
    print(f"  solo TWFE:  {[t for t in TWFE_TERMS if t not in d_local]}")

# Primeras diferencias y estadísticos suficientes

In [ ]:
# Diferencias por calendario de y y de cada término + Mundlak de las diferencias (media por PDV de cada Δtérmino: absorbe la pendiente propia del cliente) + efecto de mes
prev_cols = [sf.col('id_cliente'), (sf.col('period_id') + 1).alias('period_id')] + \
            [sf.col(t).alias(f'{t}__prev') for t in ['log_y'] + TERMS_ALL]
prev = panel.select(*prev_cols)
panel_fd = panel.join(prev, ['id_cliente', 'period_id'], 'inner')
panel_fd = panel_fd.withColumn('d_y', sf.col('log_y') - sf.col('log_y__prev'))
for t in TERMS_ALL:
    panel_fd = panel_fd.withColumn(f'd_{t}', sf.col(t) - sf.col(f'{t}__prev'))
D_TERMS = [f'd_{t}' for t in TERMS_ALL]

wId = Window.partitionBy('id_cliente')
DBARS = []
for d in D_TERMS:
    panel_fd = panel_fd.withColumn(f'{d}__bar', sf.avg(d).over(wId))
    DBARS.append(f'{d}__bar')

panel_fd = panel_fd.select('id_cliente', 'period_id', 'd_y', *D_TERMS, *DBARS) \
                   .persist(StorageLevel.MEMORY_AND_DISK)
K = len(D_TERMS)
print(f"Diseño FD: {K} Δtérminos + {K} Mundlak de diferencias + efecto de mes")

In [ ]:
# Compresión exacta del panel: Gramian distribuido de A = [1 | Δz | Δzbar | meses | d_y]
stats = panel_fd.agg(*([sf.avg(c).alias(f'm_{c}') for c in D_TERMS + DBARS] +
                       [sf.stddev_pop(c).alias(f's_{c}') for c in D_TERMS + DBARS])).collect()[0]
MU = {c: float(stats[f'm_{c}']) for c in D_TERMS + DBARS}
SD = {c: (float(stats[f's_{c}']) or 1.0) for c in D_TERMS + DBARS}
for c in SD:
    if SD[c] == 0:
        SD[c] = 1.0

MESES = sorted(r[0] for r in panel_fd.select('period_id').distinct().collect())
L_MES = len(MESES)

cols_std = [((sf.col(c) - sf.lit(MU[c])) / sf.lit(SD[c])).alias(f'z_{c}') for c in D_TERMS + DBARS]
cols_mes = [sf.when(sf.col('period_id') == sf.lit(m), 1.0).otherwise(0.0).alias(f'mes_{j}')
            for j, m in enumerate(MESES)]
df_A = panel_fd.select(sf.lit(1.0).alias('uno'), *cols_std, *cols_mes, sf.col('d_y'))

n_fd = df_A.count()
t0 = time.time()
GRAM = RowMatrix(df_A.rdd.map(lambda r: Vectors.dense(list(r)))).computeGramianMatrix().toArray()
panel_fd.unpersist()
print(f"Gramian {GRAM.shape} en {time.time()-t0:.0f}s | n diferencias: {n_fd:,}")

P  = GRAM.shape[0] - 1
G  = GRAM[:P, :P]
b  = GRAM[:P, P]
yy = float(GRAM[P, P])
assert not np.isnan(G).any() and not np.isnan(b).any(), "NaN en los estadísticos suficientes"
xs_beta = np.array([SD[c] for c in D_TERMS])   # para des-estandarizar beta al reportar

# Salvaguarda de identificación

In [ ]:
# Un término con SE anómalo (colineal con los efectos fijos) se excluye y se recalculan los estadísticos
ridge = 1e-8 * np.trace(G) / P
theta_ols = np.linalg.solve(G + ridge * np.eye(P), b)
sig2 = max((yy - 2 * theta_ols @ b + theta_ols @ (G @ theta_ols)) / n_fd, 1e-8)
se_ols = np.sqrt(np.abs(np.diag(sig2 * np.linalg.inv(G + ridge * np.eye(P)))))
se_nat = se_ols[1:1 + K] / xs_beta
DROP = [i for i in range(K) if se_nat[i] > UMBRAL_SE]
if DROP:
    print(f"Términos no identificados (SE>{UMBRAL_SE}), se excluyen: {[TERMS_ALL[i] for i in DROP]}")
    keep = [i for i in range(P)
            if not (1 <= i < 1 + K and (i - 1) in DROP)
            and not (1 + K <= i < 1 + 2 * K and (i - 1 - K) in DROP)]
    G = G[np.ix_(keep, keep)]; b = b[keep]
    TERMS_ALL = [t for i, t in enumerate(TERMS_ALL) if i not in DROP]
    D_TERMS   = ['d_' + t.replace(':', '__') for t in TERMS_ALL]
    xs_beta = np.delete(xs_beta, DROP)
    K = len(TERMS_ALL); P = len(keep)
    theta_ols = np.linalg.solve(G + ridge * np.eye(P), b)
    sig2 = max((yy - 2 * theta_ols @ b + theta_ols @ (G @ theta_ols)) / n_fd, 1e-8)
else:
    print("Todos los términos identificados")

# Modelo oficial: NUTS sobre las diferencias

In [ ]:
# Verosimilitud exacta desde los estadísticos suficientes; priors estandarizados N(0,1) y HalfNormal(1)
Gj = jnp.asarray(G)
bj = jnp.asarray(b)
dy_mean = float(b[0] / n_fd)

def hacer_modelo(escala_prior):
    def modelo():
        alpha   = numpyro.sample('alpha', dist.Normal(dy_mean, 5.0))
        beta    = numpyro.sample('beta',  dist.Normal(0.0, escala_prior).expand([K]).to_event(1))
        gamma   = numpyro.sample('gamma', dist.Normal(0.0, escala_prior).expand([K]).to_event(1))
        sig_m   = numpyro.sample('sigma_mes', dist.HalfNormal(escala_prior))
        raw_m   = numpyro.sample('mes_raw', dist.Normal(0.0, 1.0).expand([L_MES]).to_event(1))
        sigma_y = numpyro.sample('sigma_y', dist.HalfNormal(1.0))
        theta = jnp.concatenate([jnp.array([alpha]), beta, gamma, sig_m * raw_m])
        quad = yy - 2.0 * jnp.dot(theta, bj) + jnp.dot(theta, Gj @ theta)
        numpyro.factor('loglik', -0.5 * n_fd * jnp.log(2 * jnp.pi * sigma_y ** 2) - 0.5 * quad / sigma_y ** 2)
    return modelo

init_vals = {'alpha': jnp.asarray(theta_ols[0]),
             'beta': jnp.asarray(theta_ols[1:1 + K]),
             'gamma': jnp.asarray(theta_ols[1 + K:1 + 2 * K]),
             'sigma_y': jnp.asarray(float(np.sqrt(sig2)))}
u0 = theta_ols[1 + 2 * K:1 + 2 * K + L_MES]
s0 = float(max(np.std(u0), 1e-3))
init_vals['sigma_mes'] = jnp.asarray(s0)
init_vals['mes_raw'] = jnp.asarray(u0 / s0)

def correr_nuts(escala_prior, etiqueta):
    for lbl, strat, seed in [('init OLS', init_to_value(values=init_vals), SEED),
                             ('init median', init_to_median(), SEED + 1),
                             ('init OLS seed2', init_to_value(values=init_vals), SEED + 2)]:
        t0 = time.time()
        mc = MCMC(NUTS(hacer_modelo(escala_prior), target_accept_prob=TARGET_ACCEPT,
                       dense_mass=True, init_strategy=strat),
                  num_warmup=N_WARMUP, num_samples=N_SAMPLES, num_chains=N_CHAINS, progress_bar=False)
        mc.run(jax.random.PRNGKey(seed), extra_fields=('diverging',))
        smry = np_summary(mc.get_samples(group_by_chain=True))
        rhat = float(np.max(smry['beta']['r_hat']))
        ess  = float(np.min(smry['beta']['n_eff']))
        ndiv = int(np.asarray(mc.get_extra_fields()['diverging']).sum())
        print(f"{etiqueta} [{lbl}] r-hat max {rhat:.3f} | ESS min {ess:,.0f} | divergencias {ndiv} | {time.time()-t0:.0f}s")
        if rhat < UMBRAL_RHAT:
            return mc, rhat, ess, ndiv
    raise AssertionError("NUTS no convergió: revisar términos colineales")

mcmc, RHAT, ESS, NDIV = correr_nuts(1.0, 'ICB base')
post = mcmc.get_samples(group_by_chain=False)
beta_nat = np.asarray(post['beta']) / xs_beta

# Efectos por combinación e HDI 95

In [ ]:
# Efectos en % de venta evaluados en el nivel típico (mismas referencias del frecuentista); bundle contemporáneo y de largo plazo con HDI 95
def hdi95(x):
    s = np.sort(x); m = int(0.95 * len(s))
    w = s[m:] - s[:len(s) - m]; i = int(np.argmin(w))
    return float(s[i]), float(s[i + m])

def pares_combo(combo):
    quads = [c for c in combo if c in CUADRATICAS]
    if quads:
        q = quads[0]
        otras = [c for c in combo if c != q]
        pio = float(np.prod([REF[c] for c in otras])) if otras else 1.0
        base = ':'.join(termino_forma(c) for c in otras)
        t1 = f'{base}:{q}' if base else q
        t2 = f'{base}:{q}_sq' if base else f'{q}_sq'
        return [(t1, pio * REF[q]), (t2, pio * REF[q] ** 2)]
    t = construir_terminos(combo)[0]
    return [(t, float(np.prod([REF[c] for c in combo])))]

def draws_efecto(pares):
    w = np.zeros(K); ok = False
    for t, pi in pares:
        if t in TERMS_ALL:
            w[TERMS_ALL.index(t)] += pi; ok = True
    return (beta_nat @ w) if ok else None

print(f"{'combinación':<52}{'efecto %':>10}{'HDI 95%':>24}{'P(>0)':>8}")
BUNDLE_PARES = []
RATIOS_HDI = {}
for combo in LISTA_C9:
    pares = pares_combo(combo)
    dr = draws_efecto(pares)
    if dr is None:
        print(f"{' x '.join(combo):<52}{'no estimable':>10}")
        continue
    lo, hi = hdi95(dr)
    m = float(dr.mean())
    RATIOS_HDI[' x '.join(combo)] = (hi - lo) / abs(m) if m != 0 else float('inf')
    print(f"{' x '.join(combo):<52}{(np.exp(m)-1)*100:>+9.2f}%   [{(np.exp(lo)-1)*100:+.2f}; {(np.exp(hi)-1)*100:+.2f}]{(dr>0).mean():>8.3f}")
    BUNDLE_PARES += pares
for lt in LAGS:
    cap = lt[:-3]
    if lt not in TERMS_ALL:
        continue
    dr = draws_efecto([(lt, REF[cap])])
    lo, hi = hdi95(dr)
    m = float(dr.mean())
    RATIOS_HDI[f'{cap} (t-1)'] = (hi - lo) / abs(m) if m != 0 else float('inf')
    marca_rev = '  (negativo: evaluar patrón de reversión)' if m < 0 else ''
    fuera = '  (fuera de agregados)' if cap in REZAGOS_FUERA_BUNDLE else ''
    print(f"{cap + ' (t-1)':<52}{(np.exp(m)-1)*100:>+9.2f}%   [{(np.exp(lo)-1)*100:+.2f}; {(np.exp(hi)-1)*100:+.2f}]{(dr>0).mean():>8.3f}{marca_rev}{fuera}")
    if cap not in REZAGOS_FUERA_BUNDLE:
        BUNDLE_PARES.append((lt, REF[cap]))

dr_bc = draws_efecto([(t, p) for t, p in BUNDLE_PARES if not t.endswith('_l1')])
dr_bl = draws_efecto(BUNDLE_PARES)
lo, hi = hdi95(dr_bc)
print(f"\nBundle contemporáneo: {(np.exp(dr_bc.mean())-1)*100:+.1f}%  HDI [{(np.exp(lo)-1)*100:+.1f}; {(np.exp(hi)-1)*100:+.1f}]")
lo, hi = hdi95(dr_bl)
print(f"Bundle largo plazo:   {(np.exp(dr_bl.mean())-1)*100:+.1f}%  HDI [{(np.exp(lo)-1)*100:+.1f}; {(np.exp(hi)-1)*100:+.1f}]")

# Convergencia con el modelo frecuentista

In [ ]:
# Mismo diseño, mismos datos, dos motores de inferencia: los números deben coincidir; una discrepancia material es señal de error de implementación, no dos opiniones
print(f"{'término':<48}{'β TWFE':>12}{'β ICB':>12}{'Δ / SE':>9}   compatibles")
alertas = 0
for k, t in enumerate(TERMS_ALL):
    d = 'd_' + t.replace(':', '__')
    if d not in TWFE_COEF:
        continue
    bt = float(TWFE_COEF[d])
    se_t = float(np.sqrt(TWFE_V[TWFE_TERMS.index(d), TWFE_TERMS.index(d)]))
    bi = float(beta_nat[:, k].mean())
    z = abs(bi - bt) / se_t if se_t > 0 else float('inf')
    comp = 'si' if z < 3 else 'REVISAR'
    if comp != 'si':
        alertas += 1
    print(f"{t:<48}{bt:>12.6f}{bi:>12.6f}{z:>9.2f}   {comp}")

w_tw = np.zeros(len(TWFE_TERMS))
for t, pi in BUNDLE_PARES:
    d = 'd_' + t.replace(':', '__')
    if d in TWFE_TERMS:
        w_tw[TWFE_TERMS.index(d)] += pi
bt = float(w_tw @ np.asarray(TWFE['coef']))
se_b = float(np.sqrt(max(w_tw @ TWFE_V @ w_tw, 0)))
lo, hi = hdi95(dr_bl)
print(f"\nBundle largo plazo  TWFE: {(np.exp(bt)-1)*100:+.1f}%  IC  [{(np.exp(bt-1.96*se_b)-1)*100:+.1f}; {(np.exp(bt+1.96*se_b)-1)*100:+.1f}]")
print(f"Bundle largo plazo  ICB:  {(np.exp(dr_bl.mean())-1)*100:+.1f}%  HDI [{(np.exp(lo)-1)*100:+.1f}; {(np.exp(hi)-1)*100:+.1f}]")
delta_pp = abs((np.exp(dr_bl.mean())-1) - (np.exp(bt)-1)) * 100
print(f"Diferencia: {delta_pp:.1f} pp | r-hat {RHAT:.3f} | términos con discrepancia: {alertas}")
print('CONVERGEN' if (delta_pp < 5 and alertas == 0) else 'REVISAR IMPLEMENTACIÓN: discrepancia material entre motores')

# Validación y robustez

In [ ]:
# Ancho de HDI: ratio ancho / |media posterior| por combinación; ratio >= 4 marca el término como no conclusivo
print(f"{'combinación / término':<52}{'ratio HDI':>10}")
FLAG_HDI = []
for nom, ratio in RATIOS_HDI.items():
    marca = '  NO CONCLUSIVO' if ratio >= 4 else ''
    if ratio >= 4:
        FLAG_HDI.append(nom)
    print(f"{nom:<52}{ratio:>10.2f}{marca}")
print(f"\nNo conclusivos (ratio >= 4): {len(FLAG_HDI)}")

In [ ]:
# Sensibilidad a los priors: re-estimación con prior difuso (3x la varianza) y concentrado (1/3); |Δ media posterior| / sd posterior base > 1 indica sensibilidad al prior
resultados_prior = {'base': beta_nat}
bundles_prior = {'base': dr_bl}
for etiqueta, escala in [('difuso', float(np.sqrt(3.0))), ('concentrado', float(np.sqrt(1.0 / 3.0)))]:
    mc_p, _, _, _ = correr_nuts(escala, f'prior {etiqueta}')
    bnat = np.asarray(mc_p.get_samples()['beta']) / xs_beta
    resultados_prior[etiqueta] = bnat
    w = np.zeros(K)
    for t, pi in BUNDLE_PARES:
        if t in TERMS_ALL:
            w[TERMS_ALL.index(t)] += pi
    bundles_prior[etiqueta] = bnat @ w

sd_base = resultados_prior['base'].std(axis=0)
print(f"\n{'término':<48}{'base':>11}{'difuso':>11}{'concentr.':>11}{'max |Δ|/sd':>11}")
max_sens = 0.0
for k, t in enumerate(TERMS_ALL):
    m0 = resultados_prior['base'][:, k].mean()
    md_ = resultados_prior['difuso'][:, k].mean()
    mc_ = resultados_prior['concentrado'][:, k].mean()
    sens = max(abs(md_ - m0), abs(mc_ - m0)) / sd_base[k] if sd_base[k] > 0 else 0.0
    max_sens = max(max_sens, sens)
    marca = '  SENSIBLE AL PRIOR' if sens > 1 else ''
    print(f"{t:<48}{m0:>11.6f}{md_:>11.6f}{mc_:>11.6f}{sens:>11.2f}{marca}")
print(f"\nSensibilidad máxima: {max_sens:.2f}  ({'algún término dominado por el prior' if max_sens > 1 else 'el posterior lo dominan los datos, no los priors'})")

plt.figure(figsize=(9, 4))
for etiqueta, dr in bundles_prior.items():
    ef = (np.exp(dr) - 1) * 100
    plt.hist(ef, bins=60, density=True, alpha=0.45, label=f'prior {etiqueta}')
plt.xlabel('Bundle largo plazo (%)'); plt.ylabel('densidad posterior')
plt.title('Sensibilidad a los priors: densidades posteriores del bundle')
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Posterior Predictive Check desde los estadísticos suficientes: se simulan ~1,000 réplicas del outcome y se compara media y varianza contra lo observado (p-value predictivo entre 0.05 y 0.95)
rng = np.random.RandomState(SEED)
N_REP = 1000
obs_mean = float(b[0] / n_fd)
obs_var  = float(yy / n_fd - obs_mean ** 2)

draws_idx = rng.choice(post['beta'].shape[0], size=N_REP, replace=False)
G0 = G[0, :]   # primera fila del Gramian: sumas de cada columna del diseño
sim_means, sim_vars = [], []
for i in draws_idx:
    theta = np.concatenate([[float(post['alpha'][i])],
                            np.asarray(post['beta'][i]),
                            np.asarray(post['gamma'][i]),
                            float(post['sigma_mes'][i]) * np.asarray(post['mes_raw'][i])])
    s = float(post['sigma_y'][i])
    fit_sum  = float(G0 @ theta)                       # suma de los valores ajustados
    fit_ss   = float(theta @ (G @ theta))              # suma de cuadrados ajustados
    mean_rep = fit_sum / n_fd + s * rng.randn() / np.sqrt(n_fd)
    cross    = s * np.sqrt(max(fit_ss, 0)) * rng.randn()
    ee       = s ** 2 * (n_fd + np.sqrt(2 * n_fd) * rng.randn())
    var_rep  = (fit_ss + 2 * cross + ee) / n_fd - mean_rep ** 2
    sim_means.append(mean_rep); sim_vars.append(var_rep)
sim_means = np.array(sim_means); sim_vars = np.array(sim_vars)

p_mean = float((sim_means > obs_mean).mean())
p_var  = float((sim_vars > obs_var).mean())
print(f"PPC media:    observado {obs_mean:+.5f} | simulado {sim_means.mean():+.5f} | p-value {p_mean:.3f} "
      f"{'OK' if 0.05 <= p_mean <= 0.95 else 'FUERA DE RANGO'}")
print(f"PPC varianza: observado {obs_var:.5f} | simulado {sim_vars.mean():.5f} | p-value {p_var:.3f} "
      f"{'OK' if 0.05 <= p_var <= 0.95 else 'FUERA DE RANGO'}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(sim_means, bins=40, alpha=0.8)
axes[0].axvline(obs_mean, color='red', lw=1.5)
axes[0].set_title(f'PPC — media (p={p_mean:.3f})')
axes[1].hist(sim_vars, bins=40, alpha=0.8)
axes[1].axvline(obs_var, color='red', lw=1.5)
axes[1].set_title(f'PPC — varianza (p={p_var:.3f})')
plt.tight_layout(); plt.show()

# Export del modelo

In [ ]:
# Export de los resultados bayesianos y del veredicto de convergencia
def _res_term(k):
    dr = beta_nat[:, k]
    lo, hi = hdi95(dr)
    return {'media': float(dr.mean()), 'sd': float(dr.std()), 'hdi_low': lo, 'hdi_high': hi}

lo_bl, hi_bl = hdi95(dr_bl)
export = {
    'bu': DECISIONES.get('bu'),
    'modelo': 'ICB en primeras diferencias (Gramian + NUTS)',
    'terms': ['d_' + t.replace(':', '__') for t in TERMS_ALL],
    'beta': {t: _res_term(k) for k, t in enumerate(TERMS_ALL)},
    'bundle_largo': {'media_pct': float((np.exp(dr_bl.mean()) - 1) * 100),
                     'hdi_pct': [float((np.exp(lo_bl) - 1) * 100), float((np.exp(hi_bl) - 1) * 100)]},
    'diagnosticos': {'r_hat_max': RHAT, 'ess_min': ESS, 'divergencias': NDIV,
                     'sensibilidad_priors_max': float(max_sens),
                     'ppc_p_media': p_mean, 'ppc_p_varianza': p_var,
                     'no_conclusivos_hdi': FLAG_HDI,
                     'convergencia_twfe_pp': float(delta_pp)},
    'fecha_generacion': pd.Timestamp.now().isoformat(),
}
RUTA_EXPORT = f"abfss://{containerName}@{storageAccountName}.dfs.core.windows.net/CTG/{BU}/Modelo/icb_fd"
(spark.createDataFrame([(json.dumps(export, indent=2,
                                    default=lambda o: o.item() if hasattr(o, 'item') else str(o)),)], ['contenido'])
      .coalesce(1)
      .write.mode('overwrite')
      .text(RUTA_EXPORT))
print(f"Export escrito: {RUTA_EXPORT}")